# Script 2 — Acoustic spaces

This notebook recreates the four final **50% acoustic-distribution figures** from the 3,612 unique phee calls.

By default it loads the accepted distance matrices, fixed two-dimensional embeddings, figure settings, selected example calls, and example WAV files supplied with the repository. It does not calculate DTW distances, retrain the VAE, or refit an embedding during an ordinary **Run All**.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
candidates = (working_directory, working_directory.parent)
PROJECT_DIR = next(
    (path for path in candidates if (path / "data").is_dir() and (path / "code").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Start this notebook from the repository root or the code directory."
    )

ACOUSTIC_DIR = PROJECT_DIR / "data" / "acoustic"
DISTANCE_DIR = ACOUSTIC_DIR / "distances"
FEATURE_DIR = ACOUSTIC_DIR / "features"
FIGURE_INPUT_DIR = ACOUSTIC_DIR / "figure_inputs"
EXAMPLE_AUDIO_DIR = ACOUSTIC_DIR / "example_audio"
RESULTS_DIR = PROJECT_DIR / "results" / "acoustic_spaces"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CALL_ORDER = ACOUSTIC_DIR / "call_order_3612.csv"

# Safe defaults: only the saved calculations are used.
RECOMPUTE_ACOUSTIC_DISTANCES = False
RECOMPUTE_EMBEDDINGS = False
MAKE_FIGURES = True

print(f"Project directory: {PROJECT_DIR}")
print("Costly acoustic calculations are disabled.")


## 1. Exact call order

Every saved matrix uses the order in `call_order_3612.csv`. The `call_id` column is therefore both a call identifier and the corresponding matrix row and column.

In [ ]:
calls = pd.read_csv(CALL_ORDER)

required_columns = {
    "call_id", "filename", "focal ID", "conspecific_ID",
    "stage", "paired_status", "pair_id",
}
missing = sorted(required_columns - set(calls.columns))
if missing:
    raise KeyError(f"The call-order table is missing columns: {missing}")
if len(calls) != 3_612:
    raise ValueError(f"Found {len(calls):,} calls; expected 3,612.")
if not np.array_equal(calls["call_id"].to_numpy(), np.arange(len(calls))):
    raise ValueError("call_id is not in the required matrix order.")
if calls["filename"].duplicated().any():
    raise ValueError("The call-order table contains duplicate filenames.")

display(calls[[
    "call_id", "filename", "focal ID", "conspecific_ID",
    "stage", "paired_status",
]].head())
print(f"Unique calls: {len(calls):,}")


## 2. Load and validate the four acoustic distances

The four representations retain different information from the same calls:

- **STP:** Euclidean distance between five principal components of traditional spectral and temporal parameters.
- **MFCC:** Euclidean distance between five principal components of MFCC measurements.
- **DTW:** exact dynamic time warping of fixed-reference 4–12 kHz spectrograms, using cosine frame distance, a 0.20 normalised warping band, and path-length normalisation.
- **VAE:** cosine distance between 32-dimensional latent means from the Chatter variational autoencoder.

The saved matrices are checked in chunks so validation does not create unnecessary full-size copies in memory.

In [ ]:
METRICS = {
    "stp": {
        "label": "Traditional acoustic-feature space",
        "source_metric": "traditional",
        "matrix": DISTANCE_DIR / "stp_3612.npy",
    },
    "mfcc": {
        "label": "MFCC acoustic space",
        "source_metric": "mfcc",
        "matrix": DISTANCE_DIR / "mfcc_3612.npy",
    },
    "dtw": {
        "label": "DTW acoustic space",
        "source_metric": "dtw",
        "matrix": DISTANCE_DIR / "dtw_3612.npy",
    },
    "vae": {
        "label": "Chatter VAE acoustic space",
        "source_metric": "chatter",
        "matrix": DISTANCE_DIR / "vae_3612.npy",
    },
}

def validate_distance_matrix(path, expected_size):
    matrix = np.load(path, mmap_mode="r")
    if matrix.shape != (expected_size, expected_size):
        raise ValueError(
            f"{path.name} has shape {matrix.shape}; "
            f"expected {(expected_size, expected_size)}."
        )
    if not np.allclose(np.diag(matrix), 0.0, atol=1e-7):
        raise ValueError(f"{path.name} does not have a zero diagonal.")

    for start in range(0, expected_size, 512):
        stop = min(start + 512, expected_size)
        rows = np.asarray(matrix[start:stop])
        if not np.isfinite(rows).all():
            raise ValueError(f"{path.name} contains a non-finite value.")
        matching_columns = np.asarray(matrix[:, start:stop]).T
        if not np.allclose(rows, matching_columns, atol=1e-7, rtol=1e-7):
            raise ValueError(f"{path.name} is not symmetric.")
    return matrix

distance_matrices = {}
matrix_summary = []
for metric, details in METRICS.items():
    path = details["matrix"]
    if not path.is_file():
        raise FileNotFoundError(f"Required distance matrix is missing: {path}")
    matrix = validate_distance_matrix(path, len(calls))
    distance_matrices[metric] = matrix
    matrix_summary.append({
        "metric": metric.upper(),
        "calls": matrix.shape[0],
        "dtype": str(matrix.dtype),
        "size (MB)": round(path.stat().st_size / 1_000_000, 1),
        "symmetric": True,
        "zero diagonal": True,
    })

display(pd.DataFrame(matrix_summary))


## 3. Load the fixed figure inputs

The published geometry is read from fixed embedding tables. This prevents differences in PaCMAP or UMAP software versions from moving points between runs.

For every metric, the repository also contains:

- the exact 50% figure configuration;
- the 24 calls selected for the spectrogram border;
- all required example WAV files; and
- the observed number of calls in each animal × stage × context subset.

In [ ]:
embeddings = {}
example_calls = {}
figure_configs = {}
subset_counts = {}

for metric, details in METRICS.items():
    embedding_path = FIGURE_INPUT_DIR / f"{metric}_embedding.csv"
    examples_path = FIGURE_INPUT_DIR / f"{metric}_example_calls.csv"
    config_path = FIGURE_INPUT_DIR / f"{metric}_config.json"
    counts_path = FIGURE_INPUT_DIR / f"{metric}_subset_counts.csv"

    for path in (embedding_path, examples_path, config_path, counts_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required figure input is missing: {path}")

    embedding_table = pd.read_csv(embedding_path)
    examples = pd.read_csv(examples_path).sort_values("border_slot").reset_index(drop=True)
    config = json.loads(config_path.read_text(encoding="utf-8"))
    counts = pd.read_csv(counts_path)

    if len(embedding_table) != len(calls):
        raise ValueError(f"{embedding_path.name} has the wrong number of rows.")
    if not np.array_equal(
        embedding_table["call_id"].to_numpy(dtype=int),
        calls["call_id"].to_numpy(dtype=int),
    ):
        raise ValueError(f"{embedding_path.name} does not follow call_id order.")
    if not embedding_table["filename"].equals(calls["filename"]):
        raise ValueError(f"{embedding_path.name} does not follow filename order.")
    if len(examples) != 24 or examples["call_id"].nunique() != 24:
        raise ValueError(f"{examples_path.name} must contain 24 unique calls.")
    if not np.array_equal(examples["border_slot"].to_numpy(dtype=int), np.arange(24)):
        raise ValueError(f"{examples_path.name} does not contain border slots 0–23.")
    if not np.isclose(float(config["confidence"]), 0.50):
        raise ValueError(f"{config_path.name} is not the accepted 50% configuration.")
    if config["metric"] != details["source_metric"]:
        raise ValueError(
            f"{config_path.name} describes {config['metric']}, "
            f"not {details['source_metric']}."
        )

    # Use repository-relative paths even if an archived table contains an old path.
    examples["call_audio_path"] = examples["filename"].map(
        lambda filename: str(Path("data") / "acoustic" / "example_audio" / filename)
    )
    missing_audio = [
        relative_path for relative_path in examples["call_audio_path"]
        if not (PROJECT_DIR / relative_path).is_file()
    ]
    if missing_audio:
        raise FileNotFoundError(
            f"{metric}: {len(missing_audio)} example WAV files are missing. "
            f"First path: {missing_audio[0]}"
        )

    observed_counts = (
        calls.groupby(["focal ID", "stage", "paired_status"], observed=True, sort=True)
        .size()
        .rename("n_calls")
        .reset_index()
    )
    count_check = observed_counts.merge(
        counts,
        on=["focal ID", "stage", "paired_status"],
        suffixes=("_observed", "_saved"),
        validate="one_to_one",
    )
    if not count_check["n_calls_observed"].equals(count_check["n_calls_saved"]):
        raise ValueError(f"{counts_path.name} does not match the call table.")

    embeddings[metric] = embedding_table
    example_calls[metric] = examples
    figure_configs[metric] = config
    subset_counts[metric] = counts

figure_input_summary = pd.DataFrame([
    {
        "metric": metric.upper(),
        "embedding": figure_configs[metric]["embedding_method"],
        "calls": len(embeddings[metric]),
        "example WAVs": len(example_calls[metric]),
        "ellipse": f"{figure_configs[metric]['confidence']:.0%}",
    }
    for metric in METRICS
])
display(figure_input_summary)


## 4. Recreate the 50% acoustic-distribution figures

Panel A shows all calls, coloured by focal animal and shaped by social context. The surrounding spectrograms are the exact non-clipped, centroid-nearest examples selected for the final figures.

Panel B shows the same embedding divided by stage and social context. Each coloured outline contains 50% of the observed calls for one animal in that subset, based on the regularised empirical Mahalanobis-distance quantile. These ellipses are descriptive distributions, not confidence intervals. Large points are subset centroids; lines join the two members of each established pair.

In [ ]:
import librosa
import matplotlib.pyplot as plt
import soundfile as sf
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch, Ellipse
from scipy.signal import get_window

INDIVIDUAL_ORDER = ("Tabor", "Lola", "Odin", "Nougatti", "Wuschel", "Olympia")
INDIVIDUAL_LEGEND_ORDER = ("Tabor", "Wuschel", "Odin", "Lola", "Olympia", "Nougatti")
INDIVIDUAL_LABELS = {
    "Tabor": "Male A",
    "Lola": "Female A",
    "Wuschel": "Male B",
    "Olympia": "Female B",
    "Odin": "Male C",
    "Nougatti": "Female C",
}
INDIVIDUAL_COLORS = {
    "Tabor": "#1F78B4",
    "Lola": "#8EC7E8",
    "Odin": "#D95F02",
    "Nougatti": "#FDB863",
    "Wuschel": "#6F3B2E",
    "Olympia": "#C49A87",
}
PAIR_MEMBERS = {
    "Tabor-Lola": ("Tabor", "Lola"),
    "Odin-Nougatti": ("Odin", "Nougatti"),
    "Wuschel-Olympia": ("Wuschel", "Olympia"),
}
PAIR_COLORS = {
    "Tabor-Lola": "#1F78B4",
    "Odin-Nougatti": "#D95F02",
    "Wuschel-Olympia": "#6F3B2E",
}
CONTEXT_MARKERS = {"partner": "^", "non-partner": "s"}
PANEL_ORDER = (
    ("partner", "before"),
    ("partner", "after"),
    ("non-partner", "before"),
    ("non-partner", "after"),
)
PANEL_TITLES = {
    ("partner", "before"): "Partner · Before",
    ("partner", "after"): "Partner · After",
    ("non-partner", "before"): "Non-partner · Before",
    ("non-partner", "after"): "Non-partner · After",
}

TEXT_SCALE = 1.68
POINT_SIZE = 28.0
POINT_ALPHA = 0.88
CENTROID_SIZE = 220.0
EXAMPLE_SIZE = 76.0

def font_size(points):
    return float(points) * TEXT_SCALE

def style_embedding_axis(axis, x_limits, y_limits):
    axis.set_xlim(*x_limits)
    axis.set_ylim(*y_limits)
    axis.set_facecolor("white")
    axis.grid(False)
    axis.tick_params(labelsize=font_size(10), length=3.0, width=0.9, pad=2.0)
    for spine in axis.spines.values():
        spine.set_color("#555555")
        spine.set_linewidth(0.9)

def ellipse_parameters(points, complete_embedding, confidence=0.50):
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return None

    centroid = points.mean(axis=0)
    covariance = np.asarray(np.cov(points, rowvar=False, ddof=0), dtype=float)
    if covariance.shape != (2, 2) or not np.isfinite(covariance).all():
        return None

    isotropic_target = np.eye(2) * float(np.trace(covariance) / 2.0)
    shrinkage = min(0.70, 2.0 / (len(points) + 1.0))
    covariance = (
        (1.0 - shrinkage) * covariance
        + shrinkage * isotropic_target
    )

    embedding_span = np.maximum(np.ptp(complete_embedding, axis=0), 1e-9)
    covariance = covariance + np.diag(np.square(embedding_span * 0.006))

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.maximum(eigenvalues[order], 1e-12)
    eigenvectors = eigenvectors[:, order]

    centered = points - centroid
    squared_distances = np.einsum(
        "ij,jk,ik->i",
        centered,
        np.linalg.pinv(covariance),
        centered,
    )
    scale = float(np.sqrt(np.quantile(np.maximum(squared_distances, 0.0), confidence)))
    width = 2.0 * scale * float(np.sqrt(eigenvalues[0]))
    height = 2.0 * scale * float(np.sqrt(eigenvalues[1]))
    angle = float(np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])))
    return centroid, width, height, angle

def distribution_limits(metadata, coordinates, confidence):
    lower = coordinates.min(axis=0).astype(float)
    upper = coordinates.max(axis=0).astype(float)

    for context, stage in PANEL_ORDER:
        panel_mask = (
            metadata["paired_status"].eq(context)
            & metadata["stage"].eq(stage)
        ).to_numpy()
        for individual in INDIVIDUAL_ORDER:
            indices = np.flatnonzero(
                panel_mask & metadata["focal ID"].eq(individual).to_numpy()
            )
            ellipse = ellipse_parameters(
                coordinates[indices], coordinates, confidence=confidence
            )
            if ellipse is None:
                continue
            centroid, width, height, angle = ellipse
            radians = np.deg2rad(angle)
            x_radius = 0.5 * np.hypot(
                width * np.cos(radians), height * np.sin(radians)
            )
            y_radius = 0.5 * np.hypot(
                width * np.sin(radians), height * np.cos(radians)
            )
            lower = np.minimum(lower, centroid - (x_radius, y_radius))
            upper = np.maximum(upper, centroid + (x_radius, y_radius))

    span = np.maximum(upper - lower, 1e-6)
    padding = span * 0.035
    return (
        (float(lower[0] - padding[0]), float(upper[0] + padding[0])),
        (float(lower[1] - padding[1]), float(upper[1] + padding[1])),
    )

def fixed_dbfs_spectrogram(audio_path, display_config):
    audio, sample_rate = sf.read(audio_path, dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if int(sample_rate) != int(display_config["sample_rate_hz"]):
        raise ValueError(f"Unexpected sample rate {sample_rate} for {audio_path}")

    target_samples = int(
        round(display_config["display_max_duration_s"] * sample_rate)
    )
    if len(audio) > target_samples:
        raise ValueError(f"{audio_path} is longer than the fixed display canvas.")
    padding = target_samples - len(audio)
    audio = np.pad(audio, (padding // 2, padding - padding // 2))

    transform = librosa.stft(
        audio,
        n_fft=int(display_config["n_fft"]),
        hop_length=int(display_config["hop_length"]),
        win_length=int(display_config["win_length"]),
        window="hann",
        center=False,
        pad_mode="constant",
    )
    window_sum = get_window(
        "hann", int(display_config["win_length"]), fftbins=True
    ).sum()
    amplitude_full_scale = 2.0 * np.abs(transform) / float(window_sum)
    floor_amplitude = 10.0 ** (float(display_config["db_floor"]) / 20.0)
    dbfs = 20.0 * np.log10(np.maximum(amplitude_full_scale, floor_amplitude))
    dbfs = np.clip(
        dbfs,
        float(display_config["db_floor"]),
        float(display_config["db_ceiling"]),
    )

    frequencies = librosa.fft_frequencies(
        sr=sample_rate, n_fft=int(display_config["n_fft"])
    )
    band = (
        (frequencies >= float(display_config["frequency_min_hz"]))
        & (frequencies <= float(display_config["frequency_max_hz"]))
    )
    times = librosa.frames_to_time(
        np.arange(dbfs.shape[1]),
        sr=sample_rate,
        hop_length=int(display_config["hop_length"]),
    )
    return dbfs[band], frequencies[band], times

def thumbnail_positions():
    horizontal = np.linspace(0.045, 0.365, 6)
    vertical = np.linspace(0.18, 0.67, 6)
    width, height = 0.060, 0.090
    return (
        [((float(x), 0.77, width, height), "top") for x in horizontal]
        + [((0.395, float(y), width, height), "right") for y in vertical[::-1]]
        + [((float(x), 0.075, width, height), "bottom") for x in horizontal[::-1]]
        + [((0.015, float(y), width, height), "left") for y in vertical]
    )


In [ ]:
def render_acoustic_distribution_figure(metric):
    embedding_table = embeddings[metric]
    metadata = embedding_table.reset_index(drop=True)
    coordinates = embedding_table[["embedding_1", "embedding_2"]].to_numpy(dtype=float)
    examples = example_calls[metric]
    config = figure_configs[metric]
    confidence = float(config["confidence"])
    display_config = config["spectrogram_display_config"]
    embedding_label = str(config["embedding_method"])

    x_limits, y_limits = distribution_limits(metadata, coordinates, confidence)

    with plt.rc_context({
        "font.family": "DejaVu Sans",
        "font.size": font_size(10),
        "axes.labelcolor": "#222222",
        "xtick.color": "#333333",
        "ytick.color": "#333333",
    }):
        figure = plt.figure(figsize=(16.0, 9.5), facecolor="white")

        # Panel A: complete call space.
        overview = figure.add_axes((0.086, 0.195, 0.306, 0.575))
        for individual in INDIVIDUAL_ORDER:
            color = INDIVIDUAL_COLORS[individual]
            for context in ("partner", "non-partner"):
                indices = np.flatnonzero(
                    metadata["focal ID"].eq(individual).to_numpy()
                    & metadata["paired_status"].eq(context).to_numpy()
                )
                overview.scatter(
                    coordinates[indices, 0],
                    coordinates[indices, 1],
                    s=POINT_SIZE,
                    marker=CONTEXT_MARKERS[context],
                    c=color,
                    alpha=POINT_ALPHA,
                    linewidths=0,
                    rasterized=True,
                    zorder=2,
                )
        style_embedding_axis(overview, x_limits, y_limits)
        overview.set_xlabel(f"{embedding_label}-1", fontsize=font_size(10.5))
        overview.set_ylabel(f"{embedding_label}-2", fontsize=font_size(10.5))
        overview.xaxis.set_label_coords(0.5, 0.025)
        overview.yaxis.set_label_coords(0.020, 0.5)
        overview.tick_params(labelbottom=False, labelleft=False)
        overview.text(
            0.5, 0.985, "All calls",
            transform=overview.transAxes,
            ha="center", va="top",
            fontsize=font_size(9),
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.76, "pad": 1.5},
            zorder=10,
        )

        # Panel B: the four stage × context subsets.
        panel_positions = {
            ("partner", "before"): (0.508, 0.567, 0.234, 0.308),
            ("partner", "after"): (0.748, 0.567, 0.234, 0.308),
            ("non-partner", "before"): (0.508, 0.205, 0.234, 0.308),
            ("non-partner", "after"): (0.748, 0.205, 0.234, 0.308),
        }
        panel_axes = {}
        for context, stage in PANEL_ORDER:
            axis = figure.add_axes(panel_positions[(context, stage)])
            base_mask = (
                metadata["paired_status"].eq(context)
                & metadata["stage"].eq(stage)
            ).to_numpy()
            centroids = {}

            for individual in INDIVIDUAL_ORDER:
                indices = np.flatnonzero(
                    base_mask & metadata["focal ID"].eq(individual).to_numpy()
                )
                if not len(indices):
                    continue

                color = INDIVIDUAL_COLORS[individual]
                points = coordinates[indices]
                ellipse = ellipse_parameters(points, coordinates, confidence)
                if ellipse is not None:
                    centroid, width, height, angle = ellipse
                    axis.add_patch(Ellipse(
                        centroid, width, height, angle=angle,
                        facecolor=color, edgecolor="none", alpha=0.09, zorder=1,
                    ))
                    axis.add_patch(Ellipse(
                        centroid, width, height, angle=angle,
                        facecolor="none", edgecolor=color,
                        linewidth=1.45, alpha=0.82, zorder=3.5,
                    ))

                axis.scatter(
                    points[:, 0], points[:, 1],
                    s=POINT_SIZE,
                    marker=CONTEXT_MARKERS[context],
                    c=color,
                    alpha=POINT_ALPHA,
                    linewidths=0,
                    rasterized=True,
                    zorder=2,
                )
                centroids[individual] = points.mean(axis=0)

            for pair_id, members in PAIR_MEMBERS.items():
                if all(member in centroids for member in members):
                    axis.plot(
                        [centroids[member][0] for member in members],
                        [centroids[member][1] for member in members],
                        color=PAIR_COLORS[pair_id],
                        linewidth=2.0,
                        alpha=0.90,
                        zorder=3,
                    )
            for individual, centroid in centroids.items():
                axis.scatter(
                    [centroid[0]], [centroid[1]],
                    s=CENTROID_SIZE,
                    marker="o",
                    c=INDIVIDUAL_COLORS[individual],
                    edgecolors="#111111",
                    linewidths=1.3,
                    zorder=4,
                )

            style_embedding_axis(axis, x_limits, y_limits)
            axis.set_title(PANEL_TITLES[(context, stage)], fontsize=font_size(11.5), pad=5)
            if context == "non-partner":
                axis.set_xlabel(f"{embedding_label}-1", fontsize=font_size(10.5))
            else:
                axis.tick_params(labelbottom=False)
            if stage == "after":
                axis.tick_params(labelleft=False)
            panel_axes[(context, stage)] = axis

        figure.text(
            0.474, 0.540, f"{embedding_label}-2",
            ha="center", va="center", rotation=90, fontsize=font_size(10.5),
        )

        # The 24 saved example calls and their fixed 4–12 kHz dBFS spectrograms.
        scale_corners = {
            0: "top_left",
            5: "top_right",
            12: "bottom_right",
            17: "bottom_left",
        }
        for (_, row), (position, side) in zip(
            examples.iterrows(), thumbnail_positions()
        ):
            thumbnail = figure.add_axes(position)
            spectrogram, frequencies, times = fixed_dbfs_spectrogram(
                PROJECT_DIR / row["call_audio_path"], display_config
            )
            thumbnail.pcolormesh(
                times,
                frequencies / 1_000.0,
                spectrogram,
                shading="auto",
                cmap="magma",
                vmin=float(display_config["db_floor"]),
                vmax=float(display_config["db_ceiling"]),
                rasterized=True,
            )
            thumbnail.set_xlim(0.0, float(display_config["display_max_duration_s"]))
            thumbnail.set_ylim(
                float(display_config["frequency_min_hz"]) / 1_000.0,
                float(display_config["frequency_max_hz"]) / 1_000.0,
            )

            corner = scale_corners.get(int(row["border_slot"]))
            if corner is None:
                thumbnail.set_xticks([])
                thumbnail.set_yticks([])
            else:
                duration = float(display_config["display_max_duration_s"])
                low_khz = float(display_config["frequency_min_hz"]) / 1_000.0
                high_khz = float(display_config["frequency_max_hz"]) / 1_000.0
                thumbnail.set_xticks([0.0, duration])
                thumbnail.set_yticks([low_khz, high_khz])
                thumbnail.set_xticklabels(["0", f"{duration:.1f}"], fontsize=font_size(8.7))
                thumbnail.set_yticklabels(
                    [f"{low_khz:g}", f"{high_khz:g}"], fontsize=font_size(8.7)
                )
                thumbnail.set_xlabel("s", fontsize=font_size(8.7), labelpad=0)
                thumbnail.set_ylabel("kHz", fontsize=font_size(8.7), labelpad=-1)
                thumbnail.tick_params(length=2.5, width=0.7, pad=1.0)
                if corner.startswith("top"):
                    thumbnail.xaxis.set_ticks_position("top")
                    thumbnail.xaxis.set_label_position("top")
                    thumbnail.tick_params(axis="x", labeltop=True, labelbottom=False)
                if corner.endswith("right"):
                    thumbnail.yaxis.set_ticks_position("right")
                    thumbnail.yaxis.set_label_position("right")
                    thumbnail.tick_params(axis="y", labelright=True, labelleft=False)
                x_labels = thumbnail.get_xticklabels()
                y_labels = thumbnail.get_yticklabels()
                x_labels[0].set_horizontalalignment("left")
                x_labels[-1].set_horizontalalignment("right")
                y_labels[0].set_verticalalignment("bottom")
                y_labels[-1].set_verticalalignment("top")

            individual = str(row["focal ID"])
            color = INDIVIDUAL_COLORS[individual]
            for spine in thumbnail.spines.values():
                spine.set_visible(True)
                spine.set_color(color)
                spine.set_linewidth(1.6)

            call_id = int(row["call_id"])
            point = coordinates[call_id]
            anchor = {
                "top": (0.5, 0.0),
                "right": (0.0, 0.5),
                "bottom": (0.5, 1.0),
                "left": (1.0, 0.5),
            }[side]
            figure.add_artist(ConnectionPatch(
                xyA=anchor,
                coordsA=thumbnail.transAxes,
                xyB=(float(point[0]), float(point[1])),
                coordsB=overview.transData,
                arrowstyle="-",
                color=color,
                linewidth=0.8,
                alpha=0.42,
                clip_on=False,
                zorder=5,
            ))
            marker = CONTEXT_MARKERS[str(row["paired_status"])]
            overview.scatter(
                [point[0]], [point[1]],
                s=EXAMPLE_SIZE * 1.65,
                marker=marker,
                c="white",
                edgecolors="white",
                linewidths=1.5,
                zorder=7,
            )
            overview.scatter(
                [point[0]], [point[1]],
                s=EXAMPLE_SIZE,
                marker=marker,
                c=color,
                edgecolors="#111111",
                linewidths=1.0,
                zorder=8,
            )

        individual_handles = [
            Line2D(
                [0], [0], marker="o", linestyle="None",
                markerfacecolor=INDIVIDUAL_COLORS[individual],
                markeredgecolor="#111111",
                markersize=font_size(7),
                label=INDIVIDUAL_LABELS[individual],
            )
            for individual in INDIVIDUAL_LEGEND_ORDER
        ]
        context_handles = [
            Line2D(
                [0], [0],
                marker=CONTEXT_MARKERS[context],
                linestyle="None",
                markerfacecolor="#888888",
                markeredgecolor="none",
                markersize=font_size(7),
                label="Partner" if context == "partner" else "Non-partner",
            )
            for context in ("partner", "non-partner")
        ]
        summary_handles = [
            Line2D(
                [0], [0], marker="o", linestyle="None",
                markerfacecolor="#888888", markeredgecolor="#111111",
                markersize=font_size(8), label="Centroid",
            ),
            Line2D(
                [0], [0], color="#555555", linewidth=2.0, label="Pair link",
            ),
            Ellipse(
                (0, 0), width=1.2, height=0.7,
                facecolor="#999999", edgecolor="#555555",
                alpha=0.30, label="50% ellipse",
            ),
        ]
        figure.legend(
            handles=individual_handles,
            loc="lower right",
            bbox_to_anchor=(0.982, 0.012),
            ncol=2,
            borderaxespad=0,
            frameon=False,
            fontsize=font_size(9.2),
            handlelength=0.8,
            columnspacing=1.30,
            handletextpad=0.25,
            labelspacing=0.32,
        )
        figure.legend(
            handles=context_handles + summary_handles,
            loc="lower right",
            bbox_to_anchor=(0.795, 0.012),
            ncol=2,
            borderaxespad=0,
            frameon=False,
            fontsize=font_size(8.5),
            handlelength=1.0,
            labelspacing=0.24,
            columnspacing=0.40,
            handletextpad=0.25,
        )
        figure.text(
            0.015, 0.985, "A",
            va="top", fontsize=font_size(15), fontweight="bold",
        )
        figure.text(
            0.25, 0.985, METRICS[metric]["label"],
            ha="center", va="top", fontsize=font_size(13), fontweight="bold",
        )
        figure.text(
            0.482, 0.985, "B",
            va="top", fontsize=font_size(15), fontweight="bold",
        )
        figure.text(
            0.744, 0.985, "Spread by stage and context",
            ha="center", va="top", fontsize=font_size(13), fontweight="bold",
        )

    return figure


In [ ]:
if MAKE_FIGURES:
    saved_outputs = []
    for metric in METRICS:
        figure = render_acoustic_distribution_figure(metric)
        output_stem = RESULTS_DIR / f"acoustic_space_{metric}_distribution50_reproduced"
        png_path = output_stem.with_suffix(".png")
        figure.savefig(png_path, dpi=300, facecolor="white")
        display(figure)
        plt.close(figure)
        saved_outputs.append(png_path)
        print(f"Saved {png_path.relative_to(PROJECT_DIR)}")
else:
    print("MAKE_FIGURES is False; no figures were written.")


## 5. Optional recalculation

The accepted figures above deliberately use the archived matrices and embeddings. The switches below remain `False` by default.

STP and MFCC distances can be reconstructed directly from the included call table. New files receive a `_recomputed` suffix and cannot overwrite the accepted matrices. Exact DTW calculation and VAE training require the optional complete-audio and model-training archives and are kept outside an ordinary notebook run because they are substantially more expensive.

In [ ]:
if RECOMPUTE_ACOUSTIC_DISTANCES:
    from scipy.spatial.distance import pdist, squareform
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    traditional_columns = [
        "Dur 90% (s)", "Dur 50% (s)", "Center Freq (Hz)",
        "Freq 5% (Hz)", "Freq 25% (Hz)", "Freq 75% (Hz)",
        "Freq 95% (Hz)", "BW 50% (Hz)", "BW 90% (Hz)",
        "Avg Entropy (bits)", "Agg Entropy (bits)",
    ]
    mfcc_columns = [str(index) for index in range(1, 133)]

    for metric, feature_columns in {
        "stp": traditional_columns,
        "mfcc": mfcc_columns,
    }.items():
        values = calls[feature_columns].to_numpy(dtype=float)
        scaled = StandardScaler().fit_transform(values)
        scores = PCA(n_components=5, random_state=42).fit_transform(scaled)
        matrix = squareform(pdist(scores, metric="euclidean")).astype(np.float32)
        output_path = DISTANCE_DIR / f"{metric}_3612_recomputed.npy"
        np.save(output_path, matrix)
        print(f"Saved {output_path.relative_to(PROJECT_DIR)}")

    print(
        "The exact DTW and VAE distances were not started. "
        "They require their optional full audio/training archives."
    )
else:
    print("Using the four included distance matrices; no distances recalculated.")


In [ ]:
if RECOMPUTE_EMBEDDINGS:
    try:
        import pacmap
        import umap
    except ImportError as error:
        raise ImportError(
            "Install pacmap and umap-learn before requesting new embeddings."
        ) from error

    feature_paths = {
        "stp": FEATURE_DIR / "stp_pca_scores_3612.csv",
        "mfcc": FEATURE_DIR / "mfcc_pca_scores_3612.csv",
        "vae": FEATURE_DIR / "vae_latent_means_3612.npy",
    }

    new_embeddings = {}
    for metric in ("stp", "mfcc"):
        feature_table = pd.read_csv(feature_paths[metric])
        feature_prefix = "traditional_PC" if metric == "stp" else "mfcc_PC"
        feature_columns = [
            f"{feature_prefix}{component}" for component in range(1, 6)
        ]
        features = feature_table[feature_columns].to_numpy(dtype=np.float32)
        reducer = pacmap.PaCMAP(
            n_components=2,
            n_neighbors=30,
            MN_ratio=0.5,
            FP_ratio=2.0,
            distance="euclidean",
            random_state=42,
        )
        new_embeddings[metric] = reducer.fit_transform(features)

    vae_features = np.load(feature_paths["vae"]).astype(np.float32)
    vae_reducer = pacmap.PaCMAP(
        n_components=2,
        n_neighbors=30,
        MN_ratio=0.5,
        FP_ratio=2.0,
        distance="angular",
        random_state=42,
    )
    new_embeddings["vae"] = vae_reducer.fit_transform(vae_features)

    dtw_reducer = umap.UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.1,
        metric="precomputed",
        random_state=42,
    )
    new_embeddings["dtw"] = dtw_reducer.fit_transform(distance_matrices["dtw"])

    for metric, coordinates in new_embeddings.items():
        output = calls[[
            "call_id", "filename", "focal ID", "conspecific_ID",
            "stage", "paired_status", "pair_id",
        ]].copy()
        output["embedding_1"] = coordinates[:, 0]
        output["embedding_2"] = coordinates[:, 1]
        output_path = RESULTS_DIR / f"{metric}_embedding_recomputed.csv"
        output.to_csv(output_path, index=False)
        print(f"Saved {output_path.relative_to(PROJECT_DIR)}")
else:
    print("Using the fixed figure embeddings; no embedding was refitted.")


## Outputs

The default run writes four PNG figures to `results/acoustic_spaces/`:

- `acoustic_space_stp_distribution50_reproduced`
- `acoustic_space_mfcc_distribution50_reproduced`
- `acoustic_space_dtw_distribution50_reproduced`
- `acoustic_space_vae_distribution50_reproduced`

The corresponding files without `_reproduced` are the included reference exports.